# Topic Modeling – Project Notebook

Use this notebook for carrying out the analyses from the workshop notebook on your own subreddit data.

### Note on package installation
If you get an error about a missing package below, uncomment and run the `%pip install` line, then restart your kernel.

In [ ]:
# Package installation
#%pip install "numpy<2.0" pyLDAvis matplotlib networkx

> 💭 **Data note**: The patterns these methods surface reflect your community's discourse during the period you collected — not universal truths about language or the people posting. Keep your subreddit's context (rules, moderation, user base, time span) in mind when interpreting results.

## Loading the data

Make sure to use the preprocessed file from Week 1, with the `pp_text` column in it!

In all of the cells below, replace YOUR_FILE with the name of the files you are working with.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/YOUR_FILE_PP.csv')

In [ ]:
df.head(3)

In [ ]:
# Remove all rows that are '[removed]' or '[deleted]'
df = df.loc[~df['pp_text'].isin(['[removed]', '[deleted]' ]),:]

# Select only rows that have >3 characters in selftext
df = df.loc[df['pp_text'].str.len() > 3]

In [ ]:
from tqdm import tqdm 

lemmas_split = [lemma.split() for lemma in tqdm(df['pp_text'])]

## Creating a `Dictionary` with Gensim

In [ ]:
from gensim import corpora, models, similarities
from gensim.models.coherencemodel import CoherenceModel

# Create Dictionary 
dictionary = corpora.Dictionary(tqdm(lemmas_split))

# filter extremes and assign new ids
dictionary.filter_extremes(no_below=10, no_above=0.4)
dictionary.compactify() 

# SAVE DICT
dictionary.save('../../data/YOUR_FILE_lda.dict')

# Create Document-Term Matrix of our whole corpus 
corpus = [dictionary.doc2bow(text) for text in tqdm(lemmas_split)]

# Running a model

In [ ]:
from gensim.models.ldamodel import LdaModel

%time
lda_model = LdaModel(corpus=corpus,   # stream of document vectors or sparse matrix of shape
            id2word=dictionary,       # mapping from word IDs to words (for determining vocab size)
            num_topics=10,            # amount of topics
            random_state=100,         # seed to generate random state; useful for reproducibility
            passes=2,                 # amount of iterations/epochs 
            per_word_topics=False)    # computing most-likely topics for each word 

<a id='vis'></a>

# Visualizing a Topic Model

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()

# feed the LDA model into the pyLDAvis instance
lda_viz = gensimvis.prepare(lda_model, corpus, dictionary)
lda_viz

In [ ]:
# Save as standalone HTML - you can embed this on your website!
pyLDAvis.save_html(lda_viz, 'outputs_project/lda_topics_visualization.html')

On the left, there is a 2D plot of the "distance" between all of the topics (labeled as the Intertopic Distance Map). This plot uses a multidimensional scaling (MDS) algorithm. 
- Similar topics should appear close together on the plot; dissimilar topics should appear far apart. 
- The relative size of a topic's circle in the plot corresponds to the relative frequency of the topic in the corpus.

### Exploring topics and words
- You can scrutinize a topic more closely by clicking on its circle, or entering its number in the "selected topic" box in the upper-left (Note that, though the data used by gensim and pyLDAvis are the same, they don't use the same ID numbers for topics.)
- If you roll your mouse over a term in the bar chart on the right, the topic circles will resize in the plot on the left. This shows the strength of the relationship between the topics and the selected term.

### Salience
On the right, there is a bar chart with the top terms. When no topic is selected in the plot on the left, the bar chart shows the top-30 most **salient** terms in the corpus. A term's saliency is a measure of both how frequent the term is in the corpus and how "distinctive" it is in distinguishing between different topics.

### Probability Vs Exclusivity 
When you select a particular topic, this bar chart changes to show the top-30 most "relevant" terms for the selected topic. The relevance metric is controlled by the parameter λ, which can be adjusted with a slider above the bar chart:

* Setting λ close to 1.0 (the default) will rank the terms according to their probability within the topic.
* Setting λ close to 0.0 will rank the terms according to their "distinctiveness" or "exclusivity" within the topic. This means that terms that occur only in this topic, and do not occur in other topics.

You can move the slider between 0.0 and 1.0 to weigh term probability and exclusivity.

### Exploring the graph
The interactive visualization pyLDAvis produces is helpful for **individual** topics: you can manually select each topic to view its top most frequent and/or "relevant" terms, using different values of the λ parameter. This can help when you're trying to assign a name or "meaning" to each topic. 

It also helps you to see the **relationships** between topics: exploring the Intertopic Distance Plot can help you learn about how topics relate to each other, including potential higher-level structure between groups of topics.

### Getting insights about the model
If your topics are overlapping and clustered in one corner of the graph, your model probably has too many topics — a first hint that you might want to alter your model.

## Tweaking the data - POS tagging

In [ ]:
import warnings
warnings.simplefilter("ignore", DeprecationWarning)

import spacy
#!spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')

def POS(text, allowed_postags = ['NOUN', 'ADJ']):
    parsed = nlp(text)
    return [token.lemma_ for token in parsed if token.pos_ in allowed_postags]

In [ ]:
# This will take a long time
pos_lemmas_split = [POS(text) for text in tqdm(df['pp_text'])]

In [ ]:
import json
with open('../../data/YOUR_FILE_pos_lemmas.json', 'w' ) as write:
    json.dump(pos_lemmas_split, write)

# Uncomment the following two lines if you want to import this data again
#with open('../../data/YOUR_FILE_pos_lemmas.json') as f:
#    pos_lemmas_split = json.load(f)

In [ ]:
# turn them into a string so we can save them in our DF
str_pos_lemmas = [' '.join(t) for t in pos_lemmas_split]

In [ ]:
str_pos_lemmas[0]

In [ ]:
df['pos_lemmas'] = str_pos_lemmas

In [ ]:
df.to_csv('../../data/YOUR_FILE_pos_lemmas.csv', index=False)

Create new dictionary and corpus objects for Gensim.

In [ ]:
# Create Dictionary 
pos_dictionary = corpora.Dictionary(tqdm(pos_lemmas_split))

# filter extremes and assign new ids
pos_dictionary.filter_extremes(no_below=10, no_above=0.4)
pos_dictionary.compactify() 

# SAVE DICT
pos_dictionary.save('../../data/YOUR_FILE_pos_lda.dict')

# Create Document-Term Matrix of our whole corpus 
pos_corpus = [pos_dictionary.doc2bow(text) for text in tqdm(pos_lemmas_split)]

## Tweaking hyperparameters

`passes` controls how often we train the model on the entire corpus (i.e., epochs). To choose the number of passes, enable `logging` and set `eval_every=1` in `LdaModel`. This yields a **perplexity** score for every update: a measure of how well a probability model predicts a sample.

In [ ]:
import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(filename='../../data/gensim_project.log', filemode='w', format="%(asctime)s:%(levelname)s:%(message)s", level=logging.INFO)

lda_model_tweak = LdaModel(corpus=pos_corpus,
                           id2word=pos_dictionary,
                           num_topics=20, 
                           random_state=100,
                           eval_every=1,           
                           passes=5,
                           per_word_topics=False)

Search through the newly created "gensim_project.log" file and plot the log-likelihood. This shows how topic/word assignments reach a steady state (i.e. converge). If the curve flattens quickly, you do not need to set `passes` very high.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import re

p = re.compile(r"(-*\d+\.\d+) per-word .* (\d+\.\d+) perplexity")
matches = [p.findall(l) for l in open('../../data/gensim_project.log')]
matches = [m for m in matches if len(m) > 0]
tuples = [t[0] for t in matches]
likelihood = [float(t[0]) for t in tuples]
perplexity = [float(t[1]) for t in tuples]
iter = list(range(0, len(tuples) * 10, 10))

plt.figure(figsize=(8, 5))
plt.plot(iter, likelihood, color="black")
plt.ylabel("Log Likelihood")
plt.xlabel("Iteration")
plt.title("Topic Model Convergence")
plt.grid()

# Save plot to file
plt.tight_layout()
plt.savefig("outputs_project/topic_model_convergence.png", dpi=300)

plt.show()

<a id='coh'></a>

# Calculating Topic Coherence

**Topic Coherence** measures the degree of semantic similarity between high scoring words in a topic. A good model generates topics with *high* coherence scores.

In [ ]:
import warnings
warnings.simplefilter("ignore", DeprecationWarning)

# Compute Coherence Score
coherence_model = CoherenceModel(model=lda_model_tweak, corpus=pos_corpus, texts=tqdm(pos_lemmas_split), dictionary=pos_dictionary, coherence='c_v') 
coherence = coherence_model.get_coherence()
print('\nCoherence Score: ', coherence)

There's no hard and fast rule on what makes a good coherence score. In general, a score below 0.4 suggests the model's topics are not very internally consistent; 0.6–0.7 is good; anything higher should be treated with suspicion.

⚠️ **Warning**: Coherence is a statistical measure, not a meaning measure. A statistically coherent topic might still be uninterpretable, and a topic with slightly lower coherence might be the most analytically interesting one for your community. Whether topics are humanly meaningful is your judgment to make.

## Changing number of topics

This `compute_coherence_values()` function trains multiple LDA models, provides the models, and tells you their corresponding coherence scores.

In [ ]:
from gensim.models.ldamodel import LdaModel
from gensim.models.coherencemodel import CoherenceModel
from tqdm import tqdm

def compute_coherence_values(dictionary, corpus, texts, start=5, limit=20, step=3):
    """
    Compute c_v coherence for various number of topics
    """
    coherence_values = []
    model_list = []
    
    for num_topics in tqdm(range(start, limit, step), desc="Training LDA models"):
        # Train LDA model
        model = LdaModel(
            corpus=corpus, 
            id2word=dictionary, 
            num_topics=num_topics, 
            random_state=100,
            passes=5,
            alpha='auto'
        )
        
        # Calculate coherence
        coherence_model = CoherenceModel(
            model=model, 
            texts=texts, 
            dictionary=dictionary, 
            coherence='c_v'
        )
        
        model_list.append(model)
        coherence_values.append(coherence_model.get_coherence())
    
    return model_list, coherence_values

In [ ]:
# Can take a long time to run
model_list, coherence_values = compute_coherence_values(
    dictionary=pos_dictionary, 
    corpus=pos_corpus, 
    texts=pos_lemmas_split
)

Visualize the output of the coherence scores.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Show graph
start = 5
limit = 20
step = 3
x = range(start, limit, step)

plt.plot(x, coherence_values)
plt.xlabel("Num Topics")
plt.ylabel("Coherence Score")
plt.legend(["coherence_values"], loc='best')

# Save the plot BEFORE plt.show()
plt.tight_layout()
plt.savefig("outputs_project/lda_coherence_plot.png", dpi=300)
plt.show()

In [ ]:
# Print these coherence scores
c = 0
for m, cv in zip(x, coherence_values):
    print(f"model_list[{c}]: Num Topics = {m}, Coherence Value = {round(cv, 4)}")
    c += 1

In [ ]:
# Build a comparison table: num_topics, coherence, and top words for each model
rows = []
for i, (k, cv, model) in enumerate(zip(x, coherence_values, model_list)):
    top_words = [[w for w, _ in model.show_topic(t, topn=5)] for t in range(k)]
    rows.append({
        'k (topics)': k,
        'Coherence': round(cv, 4),
        'Sample topic words (topic 0)': ', '.join(top_words[0]),
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

If the coherence score seems to keep increasing, it generally makes sense to pick the model that gave the highest CV before dropping again. These metrics are only heuristics, though: go back to pyLDAvis with the models from `model_list` and compare which one looks better (non-overlapping bubbles spread across the chart, with distinct top words). **Interpretability** matters more than the highest coherence value.

Replace MY_MODEL below with the index of the model that achieves the best results for you.

In [ ]:
from gensim import corpora, models, similarities

# SAVE MODEL
optimal_lda_model = model_list[MY_MODEL]
optimal_lda_model.save('../../data/YOUR_FILE_pos_lda_optimal.model')

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()

# feed the LDA model into the pyLDAvis instance
lda_viz = gensimvis.prepare(optimal_lda_model, pos_corpus, pos_dictionary)
lda_viz

In [ ]:
# Save as standalone HTML - you can embed this on your website!
pyLDAvis.save_html(lda_viz, 'outputs_project/optimal_lda_topics_visualization.html')

In case you want to load these models from disk again:

In [ ]:
# LOAD MODEL
optimal_lda_model = LdaModel.load('../../data/YOUR_FILE_pos_lda_optimal.model')

# LOAD DICT
pos_dictionary = corpora.Dictionary.load('../../data/YOUR_FILE_pos_lda.dict')

# LOAD CORPUS
pos_corpus = [pos_dictionary.doc2bow(text) for text in tqdm(pos_lemmas_split)]

## Topic Naming as Interpretation

Assigning names to topics is the most consequential interpretive step in the pipeline. Keep in mind:

- **Names are not neutral.** The same word list can support very different labels, each foregrounding different assumptions.
- **Names can mislead.** Once you name a topic, future readers (including yourself) will see posts through that label.
- **Names are provisional.** Expect your initial names to change as you read more posts.

**Suggested workflow:**
1. Print the top 20 words per topic (see below).
2. Write a tentative name and a one-sentence justification.
3. Find 3 actual posts assigned to that topic (see the close-reading section below).
4. Revise your name based on what you actually read.

🔔 **Question**: Which topics in your model are easy to name? Which resist interpretation? What does that resistance tell you about your data?

## Naming our topics

Print out the top words of each topic, then go over all of them and give them names. Remember: the model has no semantic knowledge of the data.

In [ ]:
from pprint import pprint

# Select the ideal model and print the topics
model_topics = optimal_lda_model.show_topics(formatted=False)
pprint(optimal_lda_model.print_topics(num_words=20))

Name the topics you have. Make sure to elongate / shorten this dictionary based on how many topics you have in your final topic model!

In [ ]:
# giving names to our topics; remove or add as needed

topic_names = {0: 'NAME ME', 
               1: 'NAME ME', 
               2: 'NAME ME', 
               3: 'NAME ME', 
               4: 'NAME ME', 
               5: 'NAME ME', 
               6: 'NAME ME',
               7: 'NAME ME', 
               8: 'NAME ME', 
               9: 'NAME ME', 
               10: 'NAME ME', 
               11: 'NAME ME', 
               12: 'NAME ME', 
               13: 'NAME ME', 
               14: 'NAME ME', 
               15: 'NAME ME'
              } 

Naming topics is a heavily iterative process, based on looking closer at particular posts (see below).

<a id='use'></a>

# Using Topic Models: What is a Reddit Post About?

In [ ]:
def dominant_topic(ldamodel=optimal_lda_model, corpus=corpus, texts=df['selftext']):
    topics_data = []
    
    # Get main topic in each document
    for i, row in enumerate(ldamodel[corpus]):
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        
        # Get the Dominant topic, Perc Contribution and Keywords for each thread
        topic_num, prop_topic = row[0]
        wp = ldamodel.show_topic(topic_num)
        topic_name = topic_names[topic_num]
        topic_keywords = ", ".join([word for word, prop in wp])
        
        topics_data.append({'Dominant_Topic': topic_num, 
                            'Dominant_Topic_Name': topic_name, 
                            'Perc_Contribution': round(prop_topic,4), 
                            'Topic_Keywords': topic_keywords})
    
    # Create DataFrame
    topics_df = pd.DataFrame(topics_data)
    
    # Add original text to the end of the output
    topics_df = pd.concat([topics_df, texts.reset_index(drop=True)], axis=1)
    
    return topics_df 

# Run function - if your text column is not called 'selftext', change it below
df_topic_keywords = dominant_topic(ldamodel=optimal_lda_model, corpus=pos_corpus, texts=df['selftext'])

# Format
df_dominant_topic = df_topic_keywords.reset_index(drop=True)
df_dominant_topic.columns = ['Dominant_Topic', 'Dominant_Topic_Name', 'Topic_Perc_Contrib', 'Keywords', 'Text']

# Show
df_dominant_topic

We can now find the posts with a dominant topic using `.loc`.

Change 'TOPIC NAME' below to a topic you have named above!

In [ ]:
df_dominant_topic.loc[df_dominant_topic['Dominant_Topic_Name'] == 'TOPIC NAME']['Text']

## Close Reading by Topic

Before you finalize your topic names, read actual posts. The function below retrieves the top-N posts most dominated by a given topic, so you can read the texts the model is most confident about. Remember: the "dominant topic" for a post may only account for 40–60% of its content — the rest belongs to other topics.

🔔 **Question**: Read 3 posts from one topic. Do they all feel like they belong together? What is the common thread — and what is *not* captured by the topic name you gave?

In [ ]:
def close_read_topic(topic_name, df_dominant, original_df, top_n=3):
    """Print the top-N posts most dominated by a given topic name."""
    subset = df_dominant[df_dominant['Dominant_Topic_Name'] == topic_name].copy()
    subset = subset.sort_values('Topic_Perc_Contrib', ascending=False).head(top_n)
    
    for rank, (_, row) in enumerate(subset.iterrows(), 1):
        print(f"\n{'='*70}")
        print(f"Rank {rank} | Dominance: {row['Topic_Perc_Contrib']:.2%}")
        # Flair is only printed if your data has a 'flair_text' column
        if 'flair_text' in original_df.columns and row.name in original_df.index:
            print(f"Flair: {original_df.loc[row.name, 'flair_text']}")
        print(f"{'-'*70}")
        # Print first 600 chars for readability
        print(row['Text'][:600] + '...' if len(row['Text']) > 600 else row['Text'])

# Change 'TOPIC NAME' below to one of your own topic names
close_read_topic('TOPIC NAME', df_dominant_topic, df)

## Adding topics to original DF

Once we are happy with your topic names, we can add the dominant topic names to our original DataFrame and save it.

In [ ]:
# Use .values so the assignment ignores the filtered DataFrame's index
df['dom_topic'] = df_dominant_topic['Dominant_Topic_Name'].values
df['dom_topic_num'] = df_dominant_topic['Dominant_Topic'].values

df.to_csv('../../data/YOUR_FILE_pos_lemmas_topics.csv', index=False)

In [ ]:
df.head(3)

# 💭 Reflection: The Hermeneutics of Topic Modeling

Topic modeling surfaces patterns — but patterns are not interpretations. The meaning-making happens when you:

1. **Name** a topic (and own the assumptions embedded in that name)
2. **Read** representative posts to test whether the name holds
3. **Examine** unexpected co-occurrences that complicate your categories
4. **Ask** whose voice is centered and whose is missing

Not every topic will yield a clean insight. When a topic resists easy naming, ask what it would mean if that cluster *is* coherent — and you simply lack the right conceptual vocabulary to describe it yet.

🔔 **Question**: What would you change about how your data was preprocessed if you were designing a topic model to study a specific phenomenon in your community — rather than just topic themes? What would you keep? What would you discard?

# Topic Co-occurrence

We only added the "dominant topic" (i.e., the topic with the highest probability) to our DataFrame, but we should remember that topic models assign probabilities for all topics across all documents. 

This means we could also create a network graph that displays co-occurring topics. This is especially helpful if you have named your topics, as it allows you to see which of the themes frequently seem to occur together.

In [ ]:
# Number of topics
num_topics = optimal_lda_model.num_topics

# Initialize the overlap matrix
overlap_matrix = np.zeros((num_topics, num_topics))

# Iterate through documents and get topic probabilities
for document in tqdm(pos_corpus, desc="Processing documents"):
    document_topics = optimal_lda_model.get_document_topics(document, minimum_probability=0)
    # Create a full topic distribution for the document
    full_topic_distribution = [0] * num_topics
    for topic_num, prob in document_topics:
        full_topic_distribution[topic_num] = prob

    # Iterate through pairs of topics and add probabilities to the overlap matrix
    for i in range(num_topics):
        for j in range(num_topics):
            overlap_matrix[i, j] += full_topic_distribution[i] * full_topic_distribution[j]

# Normalize the overlap matrix by dividing by the number of documents
overlap_matrix /= len(pos_corpus)

# Zero out the diagonal (a topic always co-occurs with itself)
np.fill_diagonal(overlap_matrix, 0)

# Now apply the threshold
co_occurrence_matrix = np.where(overlap_matrix > 0.01, overlap_matrix, 0)

In [ ]:
import networkx as nx

# Create a graph from the co-occurrence matrix
G = nx.Graph()
for i in range(co_occurrence_matrix.shape[0]):
    for j in range(co_occurrence_matrix.shape[1]):
        if co_occurrence_matrix[i, j] > 0:
            G.add_edge(i, j, weight=co_occurrence_matrix[i, j])

# Define node colors based on the number of links remaining after removal
node_colors = [len(list(G.neighbors(n))) for n in G.nodes()]

# Define edge colors based on co-occurrence strength
edge_colors = [d['weight'] for _, _, d in G.edges(data=True)]

plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G)  # Can use other layouts as well

# Define an offset for the labels
label_offset = 0.05

# Create a new dictionary for the label positions
label_pos = {node: (coordinates[0], coordinates[1] + label_offset) for node, coordinates in pos.items()}

nx.draw_networkx_nodes(G, pos, node_color=node_colors, cmap=plt.cm.Reds)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, edge_cmap=plt.cm.Blues)
nx.draw_networkx_labels(G, label_pos, labels=topic_names)

# Save the figure
plt.savefig("outputs_project/networkx_cooccurrence_graph.png", dpi=300, bbox_inches='tight')

plt.show()